In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ── Load all datasets ──────────────────────────────────
matches    = pd.read_csv("../data/raw/matches.csv")
deliveries = pd.read_csv("../data/raw/deliveries.csv")
auction_22 = pd.read_csv("../data/raw/IPL_Auction_2022_FullList.csv")
auction_23 = pd.read_csv("../data/raw/IPL_2023_Auction_Sold.csv")

print("=== MATCHES ===")
print(f"Shape: {matches.shape}")
print(f"Columns: {matches.columns.tolist()}\n")

print("=== DELIVERIES ===")
print(f"Shape: {deliveries.shape}")
print(f"Columns: {deliveries.columns.tolist()}\n")

print("=== AUCTION 2022 ===")
print(f"Shape: {auction_22.shape}")
print(f"Columns: {auction_22.columns.tolist()}\n")

print("=== AUCTION 2023 ===")
print(f"Shape: {auction_23.shape}")
print(f"Columns: {auction_23.columns.tolist()}")

=== MATCHES ===
Shape: (1095, 20)
Columns: ['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']

=== DELIVERIES ===
Shape: (260920, 17)
Columns: ['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']

=== AUCTION 2022 ===
Shape: (590, 20)
Columns: ['Players', '2022 Set', 'Country', 'State Association', 'Age', 'Specialism', 'Batting Style', 'Bowling Style', 'Test caps', 'ODI caps', 'T20 caps', 'IPL', 'Previous IPLTeam(s)', '2021 Team', '2021 IPL', 'C/U/A', 'Reserve Price Rs Lakh', 'Price Paid', 'Team', 'Bid']

=== AUCTION 2023 ===
Shape: (405, 22)
Columns: ['Set No.', '2023 Set', 'First Name', 'Surname', 'Country', 'Association', 'DOB'

In [7]:
# ── Clean Matches ──────────────────────────────────────
matches["date"] = pd.to_datetime(matches["date"])
matches["season"] = matches["season"].astype(str)

# Fix team names — standardize across seasons
team_rename = {
    "Delhi Daredevils":       "Delhi Capitals",
    "Kings XI Punjab":        "Punjab Kings",
    "Deccan Chargers":        "Sunrisers Hyderabad",
    "Pune Warriors":          "Rising Pune Supergiant",
    "Rising Pune Supergiants":"Rising Pune Supergiant"
}
for col in ["team1", "team2", "toss_winner", "winner"]:
    matches[col] = matches[col].replace(team_rename)

# Add toss win = match win flag
matches["toss_match_win"] = matches["toss_winner"] == matches["winner"]

print(f"✅ Matches cleaned: {matches.shape}")
print(f"Seasons: {sorted(matches['season'].unique())}")
print(f"Teams: {sorted(matches['winner'].dropna().unique())}")

✅ Matches cleaned: (1095, 21)
Seasons: ['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024']
Teams: ['Chennai Super Kings', 'Delhi Capitals', 'Gujarat Lions', 'Gujarat Titans', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiant', 'Royal Challengers Bangalore', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']


In [15]:
# ── Clean Deliveries ───────────────────────────────────
# Merge with matches to get season info
deliveries = deliveries.merge(
    matches[["id", "season", "date"]],
    left_on="match_id", right_on="id",
    how="left"
).drop(columns=["id"])

# Fix team names
for col in ["batting_team", "bowling_team"]:
    deliveries[col] = deliveries[col].replace(team_rename)

# Add boundary flag
deliveries["is_boundary"] = deliveries["batsman_runs"].isin([4, 6])
deliveries["is_six"]      = deliveries["batsman_runs"] == 6
deliveries["is_four"]     = deliveries["batsman_runs"] == 4

# Legal deliveries only (for economy calc)
deliveries["is_legal"] = ~deliveries["extras_type"].isin(["wides", "noballs"])

print(f"✅ Deliveries cleaned: {deliveries.shape}")
print(f"Sample:\n{deliveries[['season','batter','bowler','batsman_runs','is_wicket']].head()}")

✅ Deliveries cleaned: (260920, 27)
Sample:
    season       batter   bowler  batsman_runs  is_wicket
0  2007/08   SC Ganguly  P Kumar             0          0
1  2007/08  BB McCullum  P Kumar             0          0
2  2007/08  BB McCullum  P Kumar             0          0
3  2007/08  BB McCullum  P Kumar             0          0
4  2007/08  BB McCullum  P Kumar             0          0


In [21]:
print(auction_22.columns.tolist())
print(auction_22.head(2))

['player', 'country', 'age', 'role', 'base_price_lakh', 'sold_price_lakh', 'team', 'capped_status', 'season']
        player      country  age         role  base_price_lakh  \
0     R Ashwin        India   35  ALL-ROUNDER              200   
1  Trent Boult  New Zealand   32       BOWLER              200   

   sold_price_lakh team capped_status season  
0            500.0   RR        Capped   2022  
1            800.0   RR        Capped   2022  


In [25]:
# ── Clean Auction 2023 ─────────────────────────────────
auction_23_raw = pd.read_csv("../data/raw/IPL_2023_Auction_Sold.csv")
auction_23_raw["player"] = auction_23_raw["First Name"] + " " + auction_23_raw["Surname"]
auction_23_raw = auction_23_raw.rename(columns={
    "Country":       "country",
    "Age":           "age",
    "Specialism":    "role",
    "Reserve_Price": "base_price_lakh",
    "Auction_Price": "sold_price_lakh",
    "TEAM":          "team",
    "C/U/A":         "capped_status"
})
auction_23_raw["season"] = "2023"
auction_23 = auction_23_raw[
    ["player","country","age","role","base_price_lakh",
     "sold_price_lakh","team","capped_status","season"]
].dropna(subset=["sold_price_lakh"])

# ── Combine both years ─────────────────────────────────
auction = pd.concat([auction_22, auction_23], ignore_index=True)
auction["price_multiplier"] = auction["sold_price_lakh"] / auction["base_price_lakh"]
auction["is_overseas"]      = auction["country"] != "India"

print(f"✅ Auction combined: {auction.shape}")
print(f"Seasons: {auction['season'].unique()}")
print(f"\nTop 5 most expensive:")
print(auction.nlargest(5, "sold_price_lakh")[
    ["player","team","role","sold_price_lakh","season"]
].to_string(index=False))

✅ Auction combined: (610, 11)
Seasons: ['2022' '2023']

Top 5 most expensive:
         player team         role  sold_price_lakh season
     Sam Curran PBKS  ALL-ROUNDER           1850.0   2023
  Cameron Green   MI  ALL-ROUNDER           1750.0   2023
     Ben Stokes  CSK  ALL-ROUNDER           1625.0   2023
Nicholas Pooran  LSG WICKETKEEPER           1600.0   2023
   Ishan Kishan   MI WICKETKEEPER           1525.0   2022


In [29]:
import sqlite3
import os

DB_PATH = "../data/processed/ipl.db"

# Create connection
conn = sqlite3.connect(DB_PATH)

# Save all tables to SQLite
matches.to_sql("matches", conn, if_exists="replace", index=False)
deliveries.to_sql("deliveries", conn, if_exists="replace", index=False)
auction.to_sql("auction", conn, if_exists="replace", index=False)

print("✅ Database created successfully")
print(f"Location: {DB_PATH}")

# Verify tables
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"\nTables: {tables['name'].tolist()}")

# Quick record count
for table in tables['name']:
    count = pd.read_sql(f"SELECT COUNT(*) as count FROM {table}", conn)
    print(f"  {table}: {count['count'][0]:,} rows")

✅ Database created successfully
Location: ../data/processed/ipl.db

Tables: ['matches', 'deliveries', 'auction']
  matches: 1,095 rows
  deliveries: 260,920 rows
  auction: 610 rows


In [35]:
# ── SQL Query 1: Team Win Stats ────────────────────────
team_wins = pd.read_sql("""
    SELECT 
        winner as team,
        COUNT(*) as wins,
        season
    FROM matches
    WHERE winner IS NOT NULL
    AND winner != 'NA'
    GROUP BY winner, season
    ORDER BY wins DESC
""", conn)

# ── SQL Query 2: Top Batsmen ───────────────────────────
top_batsmen = pd.read_sql("""
    SELECT 
        batter,
        SUM(batsman_runs) as total_runs,
        COUNT(DISTINCT match_id) as matches,
        ROUND(SUM(batsman_runs) * 1.0 / COUNT(*) * 100, 2) as strike_rate,
        SUM(CASE WHEN batsman_runs = 4 THEN 1 ELSE 0 END) as fours,
        SUM(CASE WHEN batsman_runs = 6 THEN 1 ELSE 0 END) as sixes,
        MAX(batsman_runs) as highest_score
    FROM deliveries
    WHERE is_legal = 1
    GROUP BY batter
    HAVING matches >= 20
    ORDER BY total_runs DESC
    LIMIT 20
""", conn)

# ── SQL Query 3: Top Bowlers ───────────────────────────
top_bowlers = pd.read_sql("""
    SELECT
        bowler,
        SUM(is_wicket) as wickets,
        COUNT(DISTINCT match_id) as matches,
        ROUND(SUM(total_runs) * 6.0 / COUNT(*), 2) as economy,
        ROUND(COUNT(*) * 1.0 / NULLIF(SUM(is_wicket), 0), 2) as bowling_sr
    FROM deliveries
    WHERE is_legal = 1
    GROUP BY bowler
    HAVING matches >= 20
    ORDER BY wickets DESC
    LIMIT 20
""", conn)

# ── SQL Query 4: Venue Stats ───────────────────────────
venue_stats = pd.read_sql("""
    SELECT
        venue,
        COUNT(*) as matches_played,
        AVG(target_runs) as avg_first_innings_score,
        SUM(CASE WHEN toss_decision='bat' THEN 1 ELSE 0 END) as chose_bat,
        SUM(CASE WHEN toss_decision='field' THEN 1 ELSE 0 END) as chose_field
    FROM matches
    WHERE target_runs IS NOT NULL
    GROUP BY venue
    HAVING matches_played >= 5
    ORDER BY matches_played DESC
    LIMIT 15
""", conn)

# ── SQL Query 5: Toss Impact ───────────────────────────
toss_impact = pd.read_sql("""
    SELECT
        toss_decision,
        COUNT(*) as total,
        SUM(CASE WHEN toss_winner = winner THEN 1 ELSE 0 END) as toss_won_match,
        ROUND(SUM(CASE WHEN toss_winner = winner THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as win_pct
    FROM matches
    WHERE winner IS NOT NULL AND winner != 'NA'
    GROUP BY toss_decision
""", conn)

# Save processed data
top_batsmen.to_csv("../data/processed/top_batsmen.csv", index=False)
top_bowlers.to_csv("../data/processed/top_bowlers.csv", index=False)
team_wins.to_csv("../data/processed/team_wins.csv", index=False)
venue_stats.to_csv("../data/processed/venue_stats.csv", index=False)

print("✅ All SQL queries executed")
print(f"\nTop 5 batsmen:\n{top_batsmen[['batter','total_runs','matches','strike_rate']].head()}")
print(f"\nTop 5 bowlers:\n{top_bowlers[['bowler','wickets','matches','economy']].head()}")
print(f"\nToss impact:\n{toss_impact}")

conn.close()

ProgrammingError: Cannot operate on a closed database.

In [37]:
conn = sqlite3.connect("../data/processed/ipl.db")

# Save all analysis dataframes
top_batsmen.to_csv("../data/processed/top_batsmen.csv", index=False)
top_bowlers.to_csv("../data/processed/top_bowlers.csv", index=False)
team_wins.to_csv("../data/processed/team_wins.csv", index=False)
venue_stats.to_csv("../data/processed/venue_stats.csv", index=False)
auction.to_csv("../data/processed/auction_clean.csv", index=False)
matches.to_csv("../data/processed/matches_clean.csv", index=False)

conn.close()
print("✅ All data saved to processed folder")

✅ All data saved to processed folder
